# Session 3 - Window Functions, CTEs, Subqueries

Welcome. In this session we will learn:

- **subqueries** (SQL inside parentheses)
- **CTEs** (`WITH ... AS`) for readable steps
- **window functions** that keep detail rows

Sample tables are in database **`training`** on **your** MySQL. Use names like `training.customers`.


---
# Chapter 1 - Subqueries

A subquery is SQL inside parentheses.

Three common shapes:

1. **Scalar** - returns one value (for example average price)
2. **IN list** - returns many ids
3. **EXISTS** - checks whether at least one matching row exists

Start with the simplest: compare each product to the average price.


### Products priced above the average


In [1]:
SELECT product_id, product_name, unit_price
FROM training.products
WHERE unit_price > (SELECT AVG(unit_price) FROM training.products)
ORDER BY unit_price DESC
LIMIT 20;


+------------+-----------------------------------+------------+
| product_id | product_name                      | unit_price |
+------------+-----------------------------------+------------+
|        239 | Acer PowerEdge R660 / SKU-0239    |  655846.03 |
|        839 | Acer PowerEdge R660 / SKU-0839    |  655585.23 |
|       1199 | Acer PowerEdge R660 / SKU-1199    |  654902.16 |
|        119 | Acer PowerEdge R660 / SKU-0119    |  654400.14 |
|       1439 | Acer PowerEdge R660 / SKU-1439    |  653037.20 |
|        959 | Acer PowerEdge R660 / SKU-0959    |  652572.44 |
|       1079 | Acer PowerEdge R660 / SKU-1079    |  651707.57 |
|        719 | Acer PowerEdge R660 / SKU-0719    |  651526.14 |
|        359 | Acer PowerEdge R660 / SKU-0359    |  651491.61 |
|        479 | Acer PowerEdge R660 / SKU-0479    |  651107.81 |
|       1319 | Acer PowerEdge R660 / SKU-1319    |  650005.15 |
|        599 | Acer PowerEdge R660 / SKU-0599    |  648706.47 |
|        674 | Acer ThinkSystem SR650 / 

### Customers who placed at least one delivered order (IN)


In [2]:
SELECT customer_id, company_name, city, country
FROM training.customers
WHERE customer_id IN (
  SELECT DISTINCT customer_id
  FROM training.orders
  WHERE status = 'delivered'
    AND customer_id IS NOT NULL
)
ORDER BY company_name
LIMIT 20;


+-------------+-------------------------+-----------+---------+
| customer_id | company_name            | city      | country |
+-------------+-------------------------+-----------+---------+
|         384 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         304 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         104 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         184 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         224 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         584 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         784 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         464 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         504 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|          64 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         424 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         264 | Beacon Networks Pvt Ltd | Hyderabad | India   |
|         664 | Beacon Networks Pvt Ltd 

### Orders with no payment (NOT EXISTS)


In [3]:
SELECT o.order_id, o.order_date, o.status, o.freight
FROM training.orders o
WHERE NOT EXISTS (
  SELECT 1 FROM training.payments p WHERE p.order_id = o.order_id
)
ORDER BY o.order_date DESC
LIMIT 20;


+----------+------------+-----------+---------+
| order_id | order_date | status    | freight |
+----------+------------+-----------+---------+
|    38499 | 2024-11-30 | delivered |  283.94 |
|    36399 | 2024-11-30 | delivered |  108.53 |
|    34299 | 2024-11-30 | delivered |   98.06 |
|    32199 | 2024-11-30 | delivered |  310.06 |
|    30099 | 2024-11-30 | delivered |  246.08 |
|    27999 | 2024-11-30 | delivered |  348.26 |
|    25899 | 2024-11-30 | delivered |  337.08 |
|    23799 | 2024-11-30 | delivered |  469.15 |
|    21699 | 2024-11-30 | delivered |  435.41 |
|    19599 | 2024-11-30 | delivered |  142.44 |
|    17499 | 2024-11-30 | delivered |  451.12 |
|    15399 | 2024-11-30 | delivered |  108.75 |
|    13299 | 2024-11-30 | delivered |  204.45 |
|    11199 | 2024-11-30 | delivered |  375.58 |
|     9099 | 2024-11-30 | delivered |  377.07 |
|     6999 | 2024-11-30 | delivered |  147.56 |
|     4899 | 2024-11-30 | delivered |  200.48 |
|     2799 | 2024-11-30 | delivered |  1

<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 3 - Subqueries</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-subqueries-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-subqueries-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 2 - CTEs (WITH ... AS)

A **CTE** is a temporary named result you can reuse in the same query.

Why people like them:

- Easier to read than nested subqueries
- You can build the answer in clear steps

Picture the flow: step 1 -> step 2 -> final SELECT.


**CTEs = named steps**

| Step | Role |
|------|------|
| 1. `WITH step1 AS (...)` | First named result |
| 2. `step2 AS (...)` | Next step can read `step1` |
| 3. Final `SELECT` | Use the named steps like tables |

A CTE is a temporary named query for one statement - easier to read than nested subqueries.


### Top cities by order count using a CTE


In [4]:
WITH city_orders AS (
  SELECT c.city, c.country, COUNT(*) AS order_count
  FROM training.orders o
  JOIN training.customers c ON c.customer_id = o.customer_id
  GROUP BY c.city, c.country
)
SELECT city, country, order_count
FROM city_orders
WHERE order_count >= 200
ORDER BY order_count DESC
LIMIT 25;


+------------+-----------+-------------+
| city       | country   | order_count |
+------------+-----------+-------------+
| Austin     | USA       |        1958 |
| Chicago    | USA       |        1958 |
| Hyderabad  | India     |        1958 |
| Chennai    | India     |        1958 |
| Pune       | India     |        1958 |
| Dubai      | UAE       |        1958 |
| Abu Dhabi  | UAE       |        1958 |
| Singapore  | Singapore |        1958 |
| Delhi      | India     |        1957 |
| Frankfurt  | Germany   |        1957 |
| Berlin     | Germany   |        1957 |
| London     | UK        |        1957 |
| New York   | USA       |        1957 |
| Jaipur     | India     |        1957 |
| Ahmedabad  | India     |        1957 |
| Kolkata    | India     |        1957 |
| Bengaluru  | India     |        1957 |
| Mumbai     | India     |        1957 |
| Kochi      | India     |        1566 |
| Manchester | UK        |        1566 |
+------------+-----------+-------------+


### Two-step CTE: monthly volume, then months above average

Real question: **Which months were busier than a typical month?**

```
Step 1 - monthly          Step 2 - above_avg
------------------        -----------------------
year | month | count      keep only months where
2023 |   3   | 1798       count > average(count)
2023 |   1   | 1797
...
```

Both steps are **CTEs**. No window function yet.


In [5]:
WITH monthly AS (
  SELECT EXTRACT(YEAR FROM order_date) AS order_year,
         EXTRACT(MONTH FROM order_date) AS order_month,
         COUNT(*) AS order_count
  FROM training.orders
  GROUP BY EXTRACT(YEAR FROM order_date), EXTRACT(MONTH FROM order_date)
),
above_avg AS (
  SELECT order_year, order_month, order_count
  FROM monthly
  WHERE order_count > (SELECT AVG(order_count) FROM monthly)
)
SELECT order_year, order_month, order_count
FROM above_avg
ORDER BY order_count DESC, order_year, order_month;


+------------+-------------+-------------+
| order_year | order_month | order_count |
+------------+-------------+-------------+
|       2023 |           3 |        1798 |
|       2023 |           1 |        1797 |
|       2023 |           5 |        1767 |
|       2023 |           7 |        1767 |
|       2023 |           8 |        1767 |
|       2023 |          10 |        1767 |
|       2023 |          12 |        1767 |
|       2024 |           1 |        1767 |
|       2024 |           3 |        1767 |
|       2024 |           5 |        1767 |
|       2024 |           7 |        1767 |
|       2024 |           8 |        1767 |
|       2024 |          10 |        1767 |
+------------+-------------+-------------+


### Optional preview: same monthly CTE, then RANK the months

Same Step 1 (`monthly`).  
Instead of filtering above-average months, label every month 1 / 2 / 3... by volume.

`RANK() OVER (...)` is a **window** function (full chapter next). Here it only adds `volume_rank`; it does not replace the CTE.


In [6]:
WITH monthly AS (
  SELECT EXTRACT(YEAR FROM order_date) AS order_year,
         EXTRACT(MONTH FROM order_date) AS order_month,
         COUNT(*) AS order_count
  FROM training.orders
  GROUP BY EXTRACT(YEAR FROM order_date), EXTRACT(MONTH FROM order_date)
)
SELECT order_year, order_month, order_count,
       RANK() OVER (ORDER BY order_count DESC) AS volume_rank
FROM monthly
ORDER BY volume_rank, order_year, order_month
LIMIT 12;


+------------+-------------+-------------+-------------+
| order_year | order_month | order_count | volume_rank |
+------------+-------------+-------------+-------------+
|       2023 |           3 |        1798 |           1 |
|       2023 |           1 |        1797 |           2 |
|       2023 |           5 |        1767 |           3 |
|       2023 |           7 |        1767 |           3 |
|       2023 |           8 |        1767 |           3 |
|       2023 |          10 |        1767 |           3 |
|       2023 |          12 |        1767 |           3 |
|       2024 |           1 |        1767 |           3 |
|       2024 |           3 |        1767 |           3 |
|       2024 |           5 |        1767 |           3 |
|       2024 |           7 |        1767 |           3 |
|       2024 |           8 |        1767 |           3 |
+------------+-------------+-------------+-------------+


### Same ranking **without** a window function

Same result idea: for each month, count how many months had a **higher** `order_count`, then add 1.

That needs a **correlated subquery** (runs “per row” against `monthly`). It works, but it is harder to read and easier to get wrong.

```
Window way (easy)              Without window (harder)
-------------------            ------------------------------
RANK() OVER (                  For each month m:
  ORDER BY order_count DESC)     count months with count > m.count
                               then rank = that_count + 1
```


In [7]:
WITH monthly AS (
  SELECT EXTRACT(YEAR FROM order_date) AS order_year,
         EXTRACT(MONTH FROM order_date) AS order_month,
         COUNT(*) AS order_count
  FROM training.orders
  GROUP BY EXTRACT(YEAR FROM order_date), EXTRACT(MONTH FROM order_date)
)
SELECT m.order_year,
       m.order_month,
       m.order_count,
       (
         SELECT COUNT(*) + 1
         FROM monthly AS m2
         WHERE m2.order_count > m.order_count
       ) AS volume_rank
FROM monthly AS m
ORDER BY volume_rank, m.order_year, m.order_month
LIMIT 12;


+------------+-------------+-------------+-------------+
| order_year | order_month | order_count | volume_rank |
+------------+-------------+-------------+-------------+
|       2023 |           3 |        1798 |           1 |
|       2023 |           1 |        1797 |           2 |
|       2023 |           5 |        1767 |           3 |
|       2023 |           7 |        1767 |           3 |
|       2023 |           8 |        1767 |           3 |
|       2023 |          10 |        1767 |           3 |
|       2023 |          12 |        1767 |           3 |
|       2024 |           1 |        1767 |           3 |
|       2024 |           3 |        1767 |           3 |
|       2024 |           5 |        1767 |           3 |
|       2024 |           7 |        1767 |           3 |
|       2024 |           8 |        1767 |           3 |
+------------+-------------+-------------+-------------+


Compare the two queries:

| Approach | How rank is written | Ease |
|----------|---------------------|------|
| Window | `RANK() OVER (ORDER BY order_count DESC)` | Short and clear |
| No window | Correlated `COUNT(*) + 1` subquery | Longer; repeats the logic per row |

Prefer the window form for ranking. The CTE is still useful for Step 1 (building `monthly`).


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 3 - CTEs</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-cte-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-cte-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 3 - Window functions

## The story (scoreboard)

Imagine a sports scoreboard for one category of products.

- **GROUP BY** is like saying: "Give me only the category totals." Detail rows disappear.
- A **window function** is like saying: "Keep every player on the board, and next to each name write their rank / running total."

```
GROUP BY (summary only)          WINDOW (detail + extra columns)
-------------------------        --------------------------------
Beverages | 12 products          Beverages | Chai      | rank 1
Condiments|  8 products          Beverages | Chang     | rank 2
                                 Beverages | Chartreuse| rank 3
                                 Condiments| Aniseed   | rank 1
```

Same idea for orders: keep each order row, and add "this is the customer's 1st / 2nd / 3rd order".

## Core pattern

```
function(...) OVER (
  PARTITION BY ...   -- optional groups (like "per customer")
  ORDER BY ...       -- order inside each group
)
```

| Piece | Everyday meaning | SQL example |
|-------|------------------|-------------|
| `OVER (...)` | "compute this across a window of rows" | required for window functions |
| `PARTITION BY` | restart the calculation for each group | `PARTITION BY customer_id` |
| `ORDER BY` inside OVER | ranking / running order inside the group | `ORDER BY order_date` |

## GROUP BY vs window (one picture)

**Starting data** - three orders for customer `1001`:

| order_id | freight |
|----------|---------|
| 10248 | 32 |
| 10249 | 11 |
| 10250 | 65 |

### Path A - `GROUP BY` (summary only)

```sql
SELECT customer_id, SUM(freight) AS total_freight
FROM orders
WHERE customer_id = 1001
GROUP BY customer_id;
```

Result: **one row** (detail disappears)

| customer_id | total_freight |
|-------------|---------------|
| 1001 | 108 |

### Path B - window `SUM` (detail kept + extra column)

```sql
SUM(freight) OVER (
  PARTITION BY customer_id
  ORDER BY order_date
)
```

Result: **three rows kept**, with a running total

| order_id | freight | running_freight | what happened |
|----------|---------|-----------------|---------------|
| 10248 | 32 | 32 | first order: 32 |
| 10249 | 11 | 43 | 32 + 11 |
| 10250 | 65 | 108 | 43 + 65 |

### Takeaway

| | GROUP BY | Window |
|--|----------|--------|
| Rows returned | 1 summary row | Same detail rows as input |
| Extra column | Totals replace the list | Totals sit beside each row |
| Best for | "What is the total?" | "What is the total so far / per row?" |



### Rank products by price inside each category


**Use case:** In each product category, which items are the most expensive?

`RANK()` gives 1 to the highest price in the category, 2 to the next, and so on.
Ties share a rank (and the next rank is skipped).


In [8]:
SELECT c.category_name, p.product_name, p.unit_price,
       RANK() OVER (
         PARTITION BY c.category_name
         ORDER BY p.unit_price DESC
       ) AS price_rank
FROM training.products p
INNER JOIN training.categories c ON c.category_id = p.category_id
ORDER BY c.category_name, price_rank, p.product_name
LIMIT 40;


+---------------+------------------------------------+------------+------------+
| category_name | product_name                       | unit_price | price_rank |
+---------------+------------------------------------+------------+------------+
| Accessories   | LG Kensington Lock / SKU-0637      |    3539.53 |          1 |
| Accessories   | LG Kensington Lock / SKU-0517      |    3537.78 |          2 |
| Accessories   | LG Kensington Lock / SKU-1237      |    3532.18 |          3 |
| Accessories   | LG Kensington Lock / SKU-0757      |    3531.35 |          4 |
| Accessories   | LG Kensington Lock / SKU-0157      |    3529.85 |          5 |
| Accessories   | LG Kensington Lock / SKU-0037      |    3528.05 |          6 |
| Accessories   | LG Kensington Lock / SKU-0997      |    3515.42 |          7 |
| Accessories   | LG Kensington Lock / SKU-1477      |    3506.17 |          8 |
| Accessories   | LG Kensington Lock / SKU-1357      |    3498.22 |          9 |
| Accessories   | LG Kensing

### RANK ties: shared rank, next number skipped

Tiny made-up list (same idea as products in one category):

| product | unit_price |
|---------|------------|
| A | 100 |
| B | 100 |
| C | 90 |
| D | 80 |

Expected `RANK()` by price descending:

| product | unit_price | price_rank | why |
|---------|------------|------------|-----|
| A | 100 | 1 | top price |
| B | 100 | 1 | tie with A - same rank |
| C | 90 | **3** | 2 is skipped because two rows already took rank 1 |
| D | 80 | 4 | next after 3 |


In [9]:
WITH prices AS (
  SELECT 'A' AS product_name, CAST(100 AS DECIMAL(10,2)) AS unit_price
  UNION ALL SELECT 'B', 100
  UNION ALL SELECT 'C', 90
  UNION ALL SELECT 'D', 80
)
SELECT product_name,
       unit_price,
       RANK() OVER (ORDER BY unit_price DESC) AS price_rank
FROM prices
ORDER BY price_rank, product_name;


+--------------+------------+------------+
| product_name | unit_price | price_rank |
+--------------+------------+------------+
| A            |     100.00 |          1 |
| B            |     100.00 |          1 |
| C            |      90.00 |          3 |
| D            |      80.00 |          4 |
+--------------+------------+------------+


### Same category price ranks **without** a window

Without `RANK()`, you compare each product to every other product in the same category with a correlated subquery.

Same answer shape - much more noise.


In [10]:
SELECT c.category_name,
       p.product_name,
       p.unit_price,
       (
         SELECT COUNT(*) + 1
         FROM training.products AS p2
         WHERE p2.category_id = p.category_id
           AND p2.unit_price > p.unit_price
       ) AS price_rank
FROM training.products AS p
INNER JOIN training.categories AS c ON c.category_id = p.category_id
ORDER BY c.category_name, price_rank, p.product_name
LIMIT 40;


+---------------+------------------------------------+------------+------------+
| category_name | product_name                       | unit_price | price_rank |
+---------------+------------------------------------+------------+------------+
| Accessories   | LG Kensington Lock / SKU-0637      |    3539.53 |          1 |
| Accessories   | LG Kensington Lock / SKU-0517      |    3537.78 |          2 |
| Accessories   | LG Kensington Lock / SKU-1237      |    3532.18 |          3 |
| Accessories   | LG Kensington Lock / SKU-0757      |    3531.35 |          4 |
| Accessories   | LG Kensington Lock / SKU-0157      |    3529.85 |          5 |
| Accessories   | LG Kensington Lock / SKU-0037      |    3528.05 |          6 |
| Accessories   | LG Kensington Lock / SKU-0997      |    3515.42 |          7 |
| Accessories   | LG Kensington Lock / SKU-1477      |    3506.17 |          8 |
| Accessories   | LG Kensington Lock / SKU-1357      |    3498.22 |          9 |
| Accessories   | LG Kensing

The window version (`RANK() OVER (PARTITION BY ... ORDER BY ...)`) states the intent in one place.
The correlated version rewrites the ranking rule by hand for every row.


### ROW_NUMBER for each customer's orders


**Story:** A support agent asks, "What was this customer's first order, and what came after?"

`ROW_NUMBER()` stamps 1, 2, 3... on each customer's orders by date.
Unlike `RANK()`, every row gets a unique number inside the partition (no ties).


In [11]:
SELECT o.customer_id, o.order_id, o.order_date, o.status,
       ROW_NUMBER() OVER (
         PARTITION BY o.customer_id
         ORDER BY o.order_date, o.order_id
       ) AS order_seq
FROM training.orders o
WHERE o.customer_id IN (1001, 1002, 1003)
ORDER BY o.customer_id, order_seq;


+-------------+----------+------------+------------+-----------+
| customer_id | order_id | order_date | status     | order_seq |
+-------------+----------+------------+------------+-----------+
|        1001 |    15401 | 2023-01-02 | processing |         1 |
|        1001 |    10601 | 2023-04-12 | processing |         2 |
|        1001 |     5801 | 2023-07-21 | processing |         3 |
|        1001 |    39401 | 2023-07-21 | processing |         4 |
|        1001 |     1001 | 2023-10-29 | processing |         5 |
|        1001 |    34601 | 2023-10-29 | processing |         6 |
|        1001 |    29801 | 2024-02-06 | processing |         7 |
|        1001 |    25001 | 2024-05-16 | processing |         8 |
|        1001 |    20201 | 2024-08-24 | processing |         9 |
|        1002 |    15402 | 2023-01-03 | shipped    |         1 |
|        1002 |    10602 | 2023-04-13 | shipped    |         2 |
|        1002 |     5802 | 2023-07-22 | shipped    |         3 |
|        1002 |    39402 

### RANK vs ROW_NUMBER on the same tie

Same four prices as before. Run **both** functions in one query:

| product | unit_price | RANK | ROW_NUMBER | difference |
|---------|------------|------|------------|------------|
| A | 100 | 1 | 1 | same |
| B | 100 | 1 | **2** | RANK ties; ROW_NUMBER never ties |
| C | 90 | **3** | 3 | RANK skipped 2; ROW_NUMBER did not need to |
| D | 80 | 4 | 4 | same |

Rule of thumb:

- Need a **competition-style place** (ties allowed)? -> `RANK()`
- Need a **unique sequence** (1st row, 2nd row, ...)? -> `ROW_NUMBER()`


In [12]:
WITH prices AS (
  SELECT 'A' AS product_name, CAST(100 AS DECIMAL(10,2)) AS unit_price
  UNION ALL SELECT 'B', 100
  UNION ALL SELECT 'C', 90
  UNION ALL SELECT 'D', 80
)
SELECT product_name,
       unit_price,
       RANK() OVER (ORDER BY unit_price DESC) AS price_rank,
       ROW_NUMBER() OVER (ORDER BY unit_price DESC, product_name) AS row_num
FROM prices
ORDER BY unit_price DESC, product_name;


+--------------+------------+------------+---------+
| product_name | unit_price | price_rank | row_num |
+--------------+------------+------------+---------+
| A            |     100.00 |          1 |       1 |
| B            |     100.00 |          1 |       2 |
| C            |      90.00 |          3 |       3 |
| D            |      80.00 |          4 |       4 |
+--------------+------------+------------+---------+


### Running freight total by day (sample month)


**Use case:** Finance wants a day-by-day running total of freight for June 2023.

`SUM(...) OVER (ORDER BY ...)` builds a cumulative total as dates increase.
Here we do not use `PARTITION BY`, so the running total is across the whole filtered set.


In [13]:
SELECT order_date, order_id, freight,
       SUM(freight) OVER (
         ORDER BY order_date, order_id
       ) AS running_freight
FROM training.orders
WHERE order_date >= DATE '2023-06-01'
  AND order_date < DATE '2023-07-01'
ORDER BY order_date, order_id
LIMIT 30;


+------------+----------+---------+-----------------+
| order_date | order_id | freight | running_freight |
+------------+----------+---------+-----------------+
| 2023-06-01 |      151 |  229.54 |          229.54 |
| 2023-06-01 |      851 |  487.07 |          716.61 |
| 2023-06-01 |     1551 |  225.93 |          942.54 |
| 2023-06-01 |     2251 |  414.83 |         1357.37 |
| 2023-06-01 |     2951 |  131.22 |         1488.59 |
| 2023-06-01 |     3651 |  233.34 |         1721.93 |
| 2023-06-01 |     4351 |  470.49 |         2192.42 |
| 2023-06-01 |     5051 |   92.78 |         2285.20 |
| 2023-06-01 |     5751 |  199.66 |         2484.86 |
| 2023-06-01 |     6451 |  101.61 |         2586.47 |
| 2023-06-01 |     7151 |  473.11 |         3059.58 |
| 2023-06-01 |     7851 |  248.98 |         3308.56 |
| 2023-06-01 |     8551 |  271.50 |         3580.06 |
| 2023-06-01 |     9251 |  406.13 |         3986.19 |
| 2023-06-01 |     9951 |  126.26 |         4112.45 |
| 2023-06-01 |    10651 |  3

### Same-category average beside each product (AVG window)


**Story:** A buyer asks, "Is this product expensive for its category?"

`AVG(unit_price) OVER (PARTITION BY category_id)` puts the category average on **every** product row.
Then you can compare `unit_price` to that average without collapsing the list.


In [14]:
SELECT c.category_name,
       p.product_name,
       p.unit_price,
       ROUND(AVG(p.unit_price) OVER (PARTITION BY p.category_id), 2) AS category_avg,
       ROUND(p.unit_price - AVG(p.unit_price) OVER (PARTITION BY p.category_id), 2) AS vs_avg
FROM training.products p
INNER JOIN training.categories c ON c.category_id = p.category_id
ORDER BY c.category_name, p.unit_price DESC
LIMIT 30;


+---------------+------------------------------------+------------+--------------+---------+
| category_name | product_name                       | unit_price | category_avg | vs_avg  |
+---------------+------------------------------------+------------+--------------+---------+
| Accessories   | LG Kensington Lock / SKU-0637      |    3539.53 |      2112.84 | 1426.69 |
| Accessories   | LG Kensington Lock / SKU-0517      |    3537.78 |      2112.84 | 1424.94 |
| Accessories   | LG Kensington Lock / SKU-1237      |    3532.18 |      2112.84 | 1419.34 |
| Accessories   | LG Kensington Lock / SKU-0757      |    3531.35 |      2112.84 | 1418.51 |
| Accessories   | LG Kensington Lock / SKU-0157      |    3529.85 |      2112.84 | 1417.01 |
| Accessories   | LG Kensington Lock / SKU-0037      |    3528.05 |      2112.84 | 1415.21 |
| Accessories   | LG Kensington Lock / SKU-0997      |    3515.42 |      2112.84 | 1402.58 |
| Accessories   | LG Kensington Lock / SKU-1477      |    3506.17 |   

### Previous order date with LAG


**Use case:** How many days between this order and the previous one for the same customer?

`LAG(order_date)` looks one row back inside the partition (after `ORDER BY order_date`).


In [15]:
SELECT customer_id,
       order_id,
       order_date,
       LAG(order_date) OVER (
         PARTITION BY customer_id
         ORDER BY order_date, order_id
       ) AS prev_order_date,
       DATEDIFF(
         order_date,
         LAG(order_date) OVER (
           PARTITION BY customer_id
           ORDER BY order_date, order_id
         )
       ) AS days_since_prev
FROM training.orders
WHERE customer_id IN (1001, 1002)
ORDER BY customer_id, order_date, order_id
LIMIT 40;


+-------------+----------+------------+-----------------+-----------------+
| customer_id | order_id | order_date | prev_order_date | days_since_prev |
+-------------+----------+------------+-----------------+-----------------+
|        1001 |    15401 | 2023-01-02 | NULL            |            NULL |
|        1001 |    10601 | 2023-04-12 | 2023-01-02      |             100 |
|        1001 |     5801 | 2023-07-21 | 2023-04-12      |             100 |
|        1001 |    39401 | 2023-07-21 | 2023-07-21      |               0 |
|        1001 |     1001 | 2023-10-29 | 2023-07-21      |             100 |
|        1001 |    34601 | 2023-10-29 | 2023-10-29      |               0 |
|        1001 |    29801 | 2024-02-06 | 2023-10-29      |             100 |
|        1001 |    25001 | 2024-05-16 | 2024-02-06      |             100 |
|        1001 |    20201 | 2024-08-24 | 2024-05-16      |             100 |
|        1002 |    15402 | 2023-01-03 | NULL            |            NULL |
|        100

## Quick map of the functions we used

| Function | Keeps detail rows? | Typical question it answers |
|----------|--------------------|-----------------------------|
| `RANK()` | Yes | Who is 1st / 2nd in this group? |
| `ROW_NUMBER()` | Yes | What is the 1st, 2nd, 3rd row in order? |
| `SUM() OVER` | Yes | What is the running total so far? |
| `AVG() OVER` | Yes | How does this row compare to its group average? |
| `LAG()` | Yes | What was the previous value? |

Remember: **window = detail rows + extra columns**. **GROUP BY = summary rows only**.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 3 - Window functions</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-windows-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s03-windows-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>
